In [2]:
# importar bibliotecas
import pandas as pd
from sqlalchemy import create_engine

db_config = {'user': 'practicum_student',         # nome de usuário
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs',  # senha
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,                       # porta de conexão
             'db': 'data-analyst-final-project-db'}  # nome do banco de dados

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                         db_config['pwd'],
                                                         db_config['host'],
                                                         db_config['port'],
                                                         db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})


Objetivo do Estudo
O objetivo deste projeto é analisar o banco de dados de um aplicativo voltado para leitores. A base reúne informações sobre livros, autores, editoras, avaliações e resenhas de usuários.
A análise busca gerar insights que apoiem a definição de uma proposta de valor para um novo produto direcionado ao público leitor.
Para isso, serão realizadas consultas SQL para responder às seguintes perguntas:

- Quantos livros foram publicados após 1º de janeiro de 2000.
- Qual a quantidade de resenhas e a nota média de cada livro.
- Qual editora publicou o maior número de livros com mais de 50 páginas.
- Qual autor possui a maior nota média, considerando apenas livros com pelo menos 50 avaliações.
- Qual a média de resenhas feitas por usuários que avaliaram mais de 50 livros.

In [5]:
query = '''
SELECT *
FROM books
LIMIT 5
'''

pd.read_sql(query, engine)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


A tabela **books** contém informações sobre os livros disponíveis na plataforma.

Principais colunas:

- book_id → identificador do livro
- author_id → autor do livro
- title → título
- num_pages → número de páginas
- publication_date → data de publicação
- publisher_id → editora

In [6]:
query = '''
SELECT *
FROM authors
LIMIT 5
'''

pd.read_sql(query, engine)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


A tabela **ratings** contém as avaliações dadas pelos usuários para cada livro.

Colunas importantes:

- rating_id
- book_id
- username
- rating

In [8]:
query = '''
SELECT *
FROM publishers
LIMIT 5
'''

pd.read_sql(query, engine)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


A tabela **publishers** contém informações sobre as editoras.

Ela se relaciona com a tabela **books** por meio da coluna `publisher_id`.

In [9]:
query = '''
SELECT *
FROM ratings
LIMIT 5
'''

pd.read_sql(query, engine)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


A tabela **ratings** contém as avaliações dadas pelos usuários para cada livro.

Colunas importantes:

- rating_id
- book_id
- username
- rating

In [7]:
query = '''
SELECT *
FROM reviews
LIMIT 5
'''

pd.read_sql(query, engine)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


A tabela **reviews** contém as resenhas escritas pelos usuários.

Ela se conecta com os livros através da coluna `book_id`.

Exercício 1
Número de livros publicados após 1º de janeiro de 2000

In [10]:
query = '''

SELECT COUNT(*) AS books_after_2000
FROM books
WHERE publication_date > '2000-01-01'

'''

pd.read_sql(query, engine)

,books_after_2000
0,819


Exercício 2
Número de resenhas e nota média para cada livro

In [11]:
query = '''

SELECT
    b.title,
    COUNT(DISTINCT rev.review_id) AS number_of_reviews,
    AVG(r.rating) AS average_rating
FROM books b
LEFT JOIN reviews rev
    ON b.book_id = rev.book_id
LEFT JOIN ratings r
    ON b.book_id = r.book_id
GROUP BY b.title
ORDER BY average_rating DESC

'''

pd.read_sql(query, engine)

,title,number_of_reviews,average_rating
0,Pop Goes the Weasel (Alex Cross #5),2,5.00
1,Angels Fall,2,5.00
2,Piercing the Darkness (Darkness #2),2,5.00
3,The Cat in the Hat and Other Dr. Seuss Favorites,0,5.00
4,Neil Gaiman's Neverwhere,2,5.00
...,...,...,...
994,The World Is Flat: A Brief History of the Twen...,3,2.25
995,Junky,2,2.00
996,Drowning Ruth,3,2.00
997,His Excellency: George Washington,2,2.00


Exercício 3
Editora com mais livros com mais de 50 páginas

In [12]:
query = '''

SELECT
    p.publisher,
    COUNT(b.book_id) AS total_books
FROM books b
JOIN publishers p
    ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY total_books DESC
LIMIT 1

'''

pd.read_sql(query, engine)

,publisher,total_books
0,Penguin Books,42


Exercício 4
Autor com maior nota média (considerando livros com pelo menos 50 avaliações)

In [13]:
query = '''

SELECT
    a.author,
    AVG(r.rating) AS avg_rating
FROM books b
JOIN authors a
    ON b.author_id = a.author_id
JOIN ratings r
    ON b.book_id = r.book_id
GROUP BY a.author, b.book_id
HAVING COUNT(r.rating) >= 50
ORDER BY avg_rating DESC
LIMIT 1

'''

pd.read_sql(query, engine)

,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.414634


Exercício 5
Número médio de resenhas entre usuários que avaliaram mais de 50 livros

In [14]:
query = '''

WITH active_users AS (

SELECT
    username
FROM ratings
GROUP BY username
HAVING COUNT(rating_id) > 50

),

user_reviews AS (

SELECT
    username,
    COUNT(review_id) AS review_count
FROM reviews
WHERE username IN (SELECT username FROM active_users)
GROUP BY username

)

SELECT
    AVG(review_count) AS avg_reviews
FROM user_reviews

'''

pd.read_sql(query, engine)

,avg_reviews
0,24.333333


Conclusão
A análise do banco de dados revelou padrões importantes tanto no comportamento dos leitores quanto nas características do catálogo de livros.
Principais insights:

- O catálogo é relativamente atual, com uma quantidade expressiva de livros publicados após 2000.
- Alguns títulos concentram grande volume de avaliações e resenhas, o que indica alto engajamento dos usuários.
- Certas editoras se destacam na publicação de livros mais extensos, com mais de 50 páginas.
- Alguns autores apresentam avaliações médias elevadas, especialmente quando considerados títulos com maior número de avaliações.
- Usuários mais ativos tendem também a escrever mais resenhas.

Esses dados podem orientar o desenvolvimento de um novo produto com foco em recomendações personalizadas, curadoria de conteúdo e estímulo à interação entre leitores.